# LAPD officer-involved shooting videos
> This notebook pulls metadata for every upload on the Los Angeles Police Department's [YouTube channel](https://www.youtube.com/@LAPDHQ/videos), filters to critical incident briefings with "OIS" in the title and fetches the closed-caption transcript for each one. It then counts cases by year and flags cases where the narrator mentions a "ghost gun".

---

#### Import Python tools and Jupyter config

In [ ]:
import json
import os
import re
import time
from pathlib import Path

import altair as alt
import jupyter_black
import pandas as pd
import requests
from googleapiclient.discovery import build
from IPython.display import Image
from youtube_transcript_api import (
    CouldNotRetrieveTranscript,
    IpBlocked,
    RequestBlocked,
    YouTubeTranscriptApi,
)
from youtube_transcript_api.proxies import GenericProxyConfig

In [ ]:
jupyter_black.load()

pd.options.display.max_columns = 100
pd.options.display.max_rows = 100
pd.options.display.max_colwidth = None

alt.data_transformers.disable_max_rows()

In [ ]:
today = pd.Timestamp("today").strftime("%Y-%m-%d")

---

## Fetch

#### Connect to the YouTube Data API with a key stored in the environment

In [ ]:
api_key = os.environ.get("YOUTUBE_KEY")
youtube = build("youtube", "v3", developerKey=api_key)

#### Get the channel's "uploads" playlist, which contains every public video

In [ ]:
channel_id = "UCager4c99nqQAmdiB7WMZhQ"  # https://www.youtube.com/@LAPDHQ

channel_response = (
    youtube.channels()
    .list(part="snippet,contentDetails,statistics", id=channel_id)
    .execute()
)

channel = channel_response["items"][0]
uploads_playlist_id = channel["contentDetails"]["relatedPlaylists"]["uploads"]

print(channel["snippet"]["title"])
print(f"Total videos: {channel['statistics']['videoCount']}")
print(f"Uploads playlist: {uploads_playlist_id}")

#### Page through the playlist, 50 videos at a time, and collect metadata for each upload

In [ ]:
videos = []
next_page_token = None

while True:
    playlist_response = (
        youtube.playlistItems()
        .list(
            part="snippet,contentDetails",
            playlistId=uploads_playlist_id,
            maxResults=50,
            pageToken=next_page_token,
        )
        .execute()
    )

    video_ids = [
        item["contentDetails"]["videoId"] for item in playlist_response["items"]
    ]

    # A second call for statistics and duration, which playlistItems doesn't return
    videos_response = (
        youtube.videos()
        .list(part="snippet,statistics,contentDetails", id=",".join(video_ids))
        .execute()
    )

    for item in videos_response["items"]:
        stats = item["statistics"]
        videos.append(
            {
                "video_id": item["id"],
                "title": item["snippet"]["title"],
                "published_at": item["snippet"]["publishedAt"],
                "duration": item["contentDetails"]["duration"],
                "views": int(stats.get("viewCount", 0)),
                "likes": int(stats.get("likeCount", 0)),
                "comments": int(stats.get("commentCount", 0)),
                "url": f"https://youtu.be/{item['id']}",
            }
        )

    next_page_token = playlist_response.get("nextPageToken")

    if not next_page_token:
        break

len(videos)

In [ ]:
all_videos = pd.DataFrame(videos)
all_videos["published_at"] = pd.to_datetime(all_videos["published_at"])
all_videos.head()

---

## Process

#### Filter to videos with "OIS" in the title. Titles follow a pattern like `Hollenbeck Area (Newton) OIS 6/9/2026 (NRF026-26)`, so we can also extract the incident date and the case number.

In [ ]:
ois = all_videos[all_videos["title"].str.contains(r"\bOIS\b", case=True)].copy()

# Incident date from the title, e.g. "6/9/2026" or "6/13/26"
ois["incident_date"] = pd.to_datetime(
    ois["title"].str.extract(r"(\d{1,2}/\d{1,2}/\d{2,4})")[0],
    format="mixed",
    errors="coerce",
)

# Case number from the title, e.g. "NRF026-26"
ois["case_number"] = ois["title"].str.extract(r"\(([A-Z]{2,3}[\s-]?\d+[-–]\d{2})\)")[0]

# Prefer the incident year; fall back to the upload year for titles without a date
ois["year"] = (
    ois["incident_date"].dt.year.fillna(ois["published_at"].dt.year).astype(int)
)

print(f"{len(ois)} OIS videos out of {len(all_videos)} uploads")
print(f"Missing incident date in title: {ois['incident_date'].isna().sum()}")
ois.head()

---

## Transcripts

#### Fetch the closed-caption transcript for each OIS video. Transcript requests don't go through the Data API and YouTube aggressively rate-limits them by IP, so they are routed through a rotating residential proxy (ScrapeOps, key in the environment as `SCRAPE_PROXY_KEY`). The cache stores the transcript text, or the failure reason for videos where captions can't be retrieved — many LAPD critical incident videos are age-restricted, which requires a signed-in session to access. Rerunning this cell only fetches videos not yet cached.

In [ ]:
cache_path = Path("data/processed/lapd_ois_transcripts.json")
cache_path.parent.mkdir(parents=True, exist_ok=True)

transcripts = json.loads(cache_path.read_text()) if cache_path.exists() else {}

proxy_url = (
    f"http://scrapeops:{os.environ['SCRAPE_PROXY_KEY']}"
    "@residential-proxy.scrapeops.io:8181"
)

transcript_api = YouTubeTranscriptApi(
    proxy_config=GenericProxyConfig(http_url=proxy_url, https_url=proxy_url)
)

# Cache values are the transcript text (str) or {"error": reason} for videos
# where captions can't be retrieved (age-restricted, no captions, etc.).
# Make repeated passes over anything still missing: the proxy rotates IPs on
# every request, so blocks and connection errors are transient — skip the
# video and catch it on the next pass.
for round_number in range(1, 6):
    to_fetch = [vid for vid in ois["video_id"] if vid not in transcripts]
    if not to_fetch:
        break
    print(f"Pass {round_number}: {len(to_fetch)} to fetch, {len(transcripts)} cached")

    for i, video_id in enumerate(to_fetch, 1):
        try:
            fetched = transcript_api.fetch(video_id)
            transcripts[video_id] = " ".join(snippet.text for snippet in fetched)
        except (RequestBlocked, IpBlocked, requests.RequestException):
            pass  # transient; leave uncached for the next pass
        except CouldNotRetrieveTranscript as e:
            transcripts[video_id] = {"error": type(e).__name__}

        if i % 25 == 0:
            print(f"{i}/{len(to_fetch)}")
            cache_path.write_text(json.dumps(transcripts))

        time.sleep(1)

    cache_path.write_text(json.dumps(transcripts))

cache_path.write_text(json.dumps(transcripts))
fetched_count = sum(isinstance(v, str) for v in transcripts.values())
print(f"Done. {fetched_count} of {len(ois)} transcripts fetched")

#### Attach the transcripts and flag videos where the narrator says "ghost gun". The status column shows why a transcript is missing, if it is.

In [ ]:
cached = ois["video_id"].map(transcripts)

ois["transcript"] = cached.apply(lambda v: v if isinstance(v, str) else None)
ois["transcript_status"] = cached.apply(
    lambda v: "ok"
    if isinstance(v, str)
    else (v["error"] if isinstance(v, dict) else "not_fetched")
)
ois["has_transcript"] = ois["transcript"].notna()

ghost_gun_pattern = re.compile(r"ghost[\s-]?gun", re.IGNORECASE)

ois["mentions_ghost_gun"] = ois["transcript"].apply(
    lambda t: bool(ghost_gun_pattern.search(t)) if isinstance(t, str) else False
)
ois["ghost_gun_mentions"] = ois["transcript"].apply(
    lambda t: len(ghost_gun_pattern.findall(t)) if isinstance(t, str) else 0
)

ois["transcript_status"].value_counts()

In [ ]:
ois[ois["mentions_ghost_gun"]][
    ["title", "year", "ghost_gun_mentions", "url"]
].sort_values("year")

---

## Aggregate

#### Cases by year, with ghost gun counts. Note: the channel began posting critical incident videos in 2018, so early years are incomplete relative to the department's full OIS history.

In [ ]:
by_year = (
    ois.groupby("year")
    .agg(
        cases=("video_id", "count"),
        with_transcript=("has_transcript", "sum"),
        ghost_gun_cases=("mentions_ghost_gun", "sum"),
    )
    .reset_index()
)

# Share of transcribed cases that mention a ghost gun
by_year["ghost_gun_share"] = (
    by_year["ghost_gun_cases"] / by_year["with_transcript"]
).round(3)

print(f"Total OIS cases: {by_year['cases'].sum()}")
print(f"With transcript: {by_year['with_transcript'].sum()}")
print(f"Ghost gun cases: {by_year['ghost_gun_cases'].sum()}")
by_year

---

## Charts

#### OIS briefing videos by year, split by whether the narrator mentions a ghost gun

In [ ]:
chart_df = ois.assign(
    category=ois["mentions_ghost_gun"].map(
        {True: "Mentions ghost gun", False: "No mention"}
    )
)

chart = (
    alt.Chart(chart_df)
    .mark_bar()
    .encode(
        x=alt.X("year:O", title=""),
        y=alt.Y("count():Q", title="Videos"),
        color=alt.Color(
            "category:N",
            title="",
            scale=alt.Scale(
                domain=["No mention", "Mentions ghost gun"],
                range=["#cccccc", "#d95f02"],
            ),
        ),
    )
    .properties(
        title="LAPD officer-involved shooting briefing videos, by year", width=650
    )
)
chart

#### Save it as a PNG and display that so it renders on GitHub

In [ ]:
Path("visuals").mkdir(exist_ok=True)
chart.save("visuals/lapd_ois_videos_by_year.png")
Image(filename="visuals/lapd_ois_videos_by_year.png")

---

## Exports

#### CSV (video-level table without the full transcript text, plus the annual summary)

In [ ]:
ois.drop(columns="transcript").to_csv(
    f"data/processed/lapd_ois_videos_{today}.csv", index=False
)
by_year.to_csv(f"data/processed/lapd_ois_videos_by_year_{today}.csv", index=False)

#### JSON (video-level table including the transcript text)

In [ ]:
ois.to_json(
    f"data/processed/lapd_ois_videos_transcripts_{today}.json",
    indent=4,
    orient="records",
    date_format="iso",
)

---

## Metadata

- **Source:** [Los Angeles Police Department on YouTube](https://www.youtube.com/@LAPDHQ/videos), channel ID `UCager4c99nqQAmdiB7WMZhQ`, via the YouTube Data API v3 (key in the environment as `YOUTUBE_KEY`) and the `youtube-transcript-api` package. Transcript requests are routed through a ScrapeOps residential proxy (`SCRAPE_PROXY_KEY`) because YouTube rate-limits caption scraping by IP.
- **Caveats:** The channel began posting critical incident briefings in 2018, so counts by year reflect videos posted, not the department's complete OIS history. Some videos cover more than one incident and a few OIS incidents may lack a posted video. Many videos are age-restricted, and their captions can't be retrieved without a signed-in session — check `transcript_status` for coverage before drawing conclusions from the transcript-based fields. Transcripts are YouTube's auto-generated captions unless the department uploaded its own, so transcription errors are possible. The "ghost gun" flag matches the phrases "ghost gun" and "ghost-gun" in the transcript text.
- **Columns:** `video_id`, `title`, `published_at` (upload timestamp), `duration` (ISO 8601), `views`, `likes`, `comments`, `url`, `incident_date` (parsed from title), `case_number` (parsed from title, e.g. NRF026-26), `year` (incident year, falling back to upload year), `transcript_status` (`ok` or the failure reason, e.g. `AgeRestricted`), `has_transcript`, `mentions_ghost_gun`, `ghost_gun_mentions` (count of matches).
- **Cache:** Transcripts are stored in `data/processed/lapd_ois_transcripts.json` keyed by video ID; delete a key (or the file) to force a refetch.